# MammoAI — обучение в Colab
Runtime → Change runtime type → **GPU (T4/L4/A100)**.

Локально выполните `sh scripts/pack_for_colab.sh` и загрузите `mammo_code.zip` в первой ячейке.

In [ ]:
from google.colab import files
uploaded = files.upload()  # выберите mammo_code.zip
!rm -rf /content/mammo && mkdir -p /content/mammo && unzip -q -o mammo_code.zip -d /content/mammo
%cd /content/mammo

In [ ]:
!pip install -q -r requirements.txt

## 1. Данные и метки
Скачивание снимков MammoGPS (CBIS-DDSM, CMMD, INbreast) и меток BI-RADS/патологии с TCIA.
Опционально: положите `INbreast.csv` в `data/sources/` до запуска `mammo.labels`.

In [ ]:
!python scripts/download_data.py
!python scripts/fetch_source_labels.py
!python -m mammo.labels
!python -m mammo.prepare_yolo

## 2. Детектор очагов (YOLO11s)

In [ ]:
!python -m train.train_detector --epochs 100 --batch 16 --device 0

## 3. Классификатор: злокачественность + BI-RADS

In [ ]:
!python -m train.train_classifier --epochs 30 --batch 32 --device cuda

## 4. Оценка на test

In [ ]:
!python -m train.evaluate --device cuda

## 5. Скачать веса
Распакуйте архив в папку `models/` проекта на Mac.

In [ ]:
!cd models && zip -q ../mammo_models.zip detector.pt classifier.pt
files.download('/content/mammo/mammo_models.zip')